EVALUATION OF AUDIO FRAME TO FRAME BY WINDOWS OF 0.04 ms with a PRECALCULATED MASK
Net: afterburner8k and test (16k)

In [16]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
from spicy import signal
import pickle
import warnings
import gzip
import scipy.io
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

#workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
workspace_dir = '/home/adelval/BTS/TFM/test/'

sys.path.append(workspace_dir + 'src/net')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [17]:
x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
#x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
print('  x_test: %s' % (x_test))

  x_test: ['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [18]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [19]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010):
    N = int(Ns * fs)            # Number of samples in each window
    M = int(Ms * fs)            # Step size (number of samples between window starts)
    n = (len(x) + M - 1) // M   # Number of frames
    print(len(x))
    print("Number of shifts", n)    
    T = (n - 1) * M + N         # Total signal length needed to fit the frames
    print(T)
    xa = x.copy()
    if T > len(x):
        xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, n * M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    return xa[ind.astype(int).T].astype(np.float32)

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[0])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=0)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


def frame_fft(data, fs, w, nfft):
    
    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]

    x = offset(data)

    # Emphasis to increase the amplitude of high freq
    x = preemphasis(x)

    XX = []

    X = hamming(x)
    print("El frame enventanado tiene dimensión: ",X.shape)
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    print("Vector con Power Spectral Density del frame: \n", X[:10])

    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [20]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


def frame_fb_mfcc(data, fs, B, w, nfft):

    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
    fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
    #print(f'fb es {len(fb[0])}')
    dct = [ f_base_dct(Bi) for Bi in B] 
    #print(f'dct es {len(dct[0])}')


    x = offset(data)
    x = preemphasis(x)
    XX = []
    for i,w in enumerate(w):
        X = hamming(x)
        
        Xfft = fft(X, nfft[i])
        Xb = np.log(Xfft.dot( fb[i] ) + 1)
        Xc = Xb.dot(dct[i])                                
        
        X = np.concatenate( [Xb, Xc], 0 )
        
        X = np.asarray(X, dtype=np.float32)
        XX.append(X)
    XX = np.concatenate(XX, 0)
    
    #print("El tamaño de x08k2 es: ",XX.shape)
    print("Vector con FilterBank MFCC del frame: \n", XX[:10])
    
    return XX

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc):
    # Normalization of fbmfcc
    file = workspace_dir + 'data/model/fe1_norm1.pkl'  # de donde salen??
    x = frame_fbmfcc
    mu, std = read_pkl(file)
    x -= mu
    x /= std + 1e-6
    print("Vector con FB MFCC normalizado del frame: \n", x[:10])

    return x
    

In [21]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append( workspace_dir + 'src/net')

from net_snr import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=False)
net_snr.load_theta( workspace_dir + 'data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:


    nb_params: 29.99M
    cuda: False
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/test/data/model/theta_last


In [22]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    print(x.shape)
    x = x.reshape(1, -1)
    print(x.shape)

    snr = net_snr.predict(x)
    print(snr[0,:])
    snr = to_numpy(snr.squeeze())
    print(f'La máscara del frame caculado es de {snr.shape}')
    #scipy.io.savemat(f, mdict={'snr': snr})
    x = to_numpy(x.squeeze())
    snr = to_numpy(snr.squeeze())
    return snr


In [23]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    #it = int(np.floor((data.size-frame)/shift))
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(Xfft.shape)
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        #print(f'La salida de la ifft tendra {outw.shape} samples') # de las que nos quedamos con 640 porque el resto son relleno
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    print(snr_net.shape)
    snr_net = snr_net.reshape(-1,1)
    print("2",snr_net.shape)
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


#frame = int(fs*w[0])
#shift = int(fs*m)


#wav = '/home/adelval/BTS/TFM/test/'+ x_test[0]
#wav = x_test[0]
#print(f'Audio seleccionado es --> {wav}')

#x = buffer_frame      # 0.04 * fs = 640 samples 
#print(f'La snr cargada es de dimensiones {snr_net.shape}')
    
#xenh, filt = noiseReduction(x, snr_net, fs, frame, shift, nfft, gmin)
#print(f'Las dimensiones del filtro son {filt.shape}')

#wavenh = wav.replace('audio','prueba')    
#if not os.path.isdir(os.path.dirname(wavenh)):
#    os.makedirs(os.path.dirname(wavenh))
#wavfile.write(wavenh,fs,xenh)
#    
#snr = int(wada_snr(xenh))
#print('snr(wada)=%idB, file: %s' % (snr, wav))


In [25]:
# Empleando una máscara precalculada, evalua el audio frame a frame con ventanas de 4 segundos

audio, fs = read_audio(x_test[0])
print(f'La frecuencia de muestreo es {fs} Hz')
# Parámetros

#fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
gmin = 0.0562


frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
shift_size = 0.01  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
window_size = 0.04  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640

# Inicialización de buffers
buffer_frame = np.zeros(0)  # Buffer de ventana recibida
accum_window = np.zeros((0, window_samples))  # Matriz vacía para almacenar las ventanas
#snr_frame_mask = np.ones((1,512)).T
snr_net_file = '/home/adelval/BTS/TFM/audios/audio_1.mat'
# snr_net_file = '/home/adelval/BTS/TFM/test/data/out/minitest_16k/7-CH0_C01_city_5dB.mat'
# snr_net_file = '/home/adelval/BTS/TFM/afterburner8k/data/out/minitest_8k/5-CH0_C01_stadium_15dB.mat'
snr_frame_mask = loadmat(snr_net_file)['snr'].T
print(f'Tamaño de la máscara {snr_frame_mask.shape}')
print(f'La mascara del primer fragmento es {snr_frame_mask[:,0]}')

yenh = np.zeros(len(audio)) # 4 segundos de audio
# Simulación de llegada de frames
for n_frame in range(465):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]
    
    # Concatenar el frame recibido al buffer `buffer_frame`
    buffer_frame = np.concatenate([buffer_frame, frame])
    
    # Si el buffer alcanza o excede las 640 muestras, se almacena en `accum_window`
    if len(buffer_frame) >= window_samples:
        # Extraer las primeras 640 muestras como ventana completa
        work_window = buffer_frame[:window_samples]
        print("Iteración número:", n_frame)
        print("Ventana acumulada", work_window[:10])
        print("Ventana acumulada", work_window[160:170])
        print("Ventana acumulada", work_window[320:330])
        print("Ventana acumulada", work_window[480:490])
        
        
        # Eliminar la ventana más antigua si ya hay 20 ventanas almacenadas
        if accum_window.shape[0] >= 20:
            accum_window = np.delete(accum_window, 0, axis=0)  # Eliminar la primera fila (más antigua)
        
        # Agregar la nueva ventana al final de `accum_window`
        accum_window = np.vstack([accum_window, work_window])
        
        # if accum_window.shape[0] >= 4:
        #     print('Inference start')
        #     frame_psd = frame_fft(accum_window, fs, w, nfft)
        #     frame_psd_log = log_scale(frame_psd)
        #     frame_fbmfcc = frame_fb_mfcc(accum_window, fs, B, w, nfft)
        #     frame_fbmfcc_norm = norm_fb_frame(frame_fbmfcc)
        #     frame_concat = np.concatenate( (frame_psd_log,frame_fbmfcc_norm), 0 )
        #     snr_frame_mask = net_eval(frame_concat)
        #     #snr_frame_mask = snr_frame_mask[:, np.newaxis]
        #     print(f'La snr cargada es de dimensiones {snr_frame_mask.shape}')
        #     print(f'La máscara de la ventana es {snr_frame_mask}')    
        

        # Aqui haría la evaluacion con la máscara pertinente( para las primeras 3 ventanas sin máscara calculada)
        print('Evaluation of window')
        x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
        print(f'El frame sin enventanado resulta {x[:10]}')
        xenh, filt = noiseReduction(x, snr_frame_mask[:,n_frame-3], fs, window_samples, shift_samples, nfft[0], gmin)
        print(f'Las dimensiones del filtro son {filt.shape}')
        cnt = n_frame - 3
        print(f'Window processed {cnt} {xenh.shape} {yenh[cnt*shift_samples : cnt*shift_samples+window_samples].shape}')
        yenh[cnt*shift_samples : cnt*shift_samples+window_samples] = yenh[cnt*shift_samples : cnt*shift_samples+window_samples] + xenh[0:window_samples]
        # Desplazar las muestras en `buffer_frame` para la próxima ventana
        buffer_frame = buffer_frame[shift_samples:]


print(f'el audio tiene len {len(yenh)}')
wavfile.write("/home/adelval/BTS/TFM/audios/enh/5-CH0_C01_stadium_15dB_8k_enh.wav",fs,yenh)

# Mostrar el resultado final
print(f'La matriz de ventanas acumuladas tiene {accum_window.shape[0]} ventanas')
print(f'1ra ventana en acumulada: {accum_window[0][:10]}')  
print(f'2da ventana en acumulada: {accum_window[1][:10]}')  
print(f'3da ventana en acumulada: {accum_window[2][:10]}')  
print(f'4da ventana en acumulada: {accum_window[3][:10]}')  
print(f'5da ventana en acumulada: {accum_window[4][:10]}')  
print(f'6da ventana en acumulada: {accum_window[5][:10]}')  
print(f'7da ventana en acumulada: {accum_window[6][:10]}')  
print(f'20da ventana en acumulada: {accum_window[19][:10]}')  
print(f'Última ventana en acumulada: {accum_window[-1][:10]}')  

La frecuencia de muestreo es 16000 Hz
Tamaño de la máscara (512, 466)
La mascara del primer fragmento es [0.33868918 0.37613457 0.3741048  0.34872591 0.36367503 0.40903547
 0.46936008 0.51463634 0.5507999  0.5669161  0.6032347  0.6130794
 0.63549405 0.6692364  0.68757457 0.71275425 0.749676   0.77683806
 0.79358053 0.8038157  0.7944289  0.8028796  0.79731554 0.7960488
 0.80437374 0.7968596  0.7954531  0.8066587  0.8023248  0.8062155
 0.8139721  0.8351213  0.8612058  0.87144166 0.87458897 0.8822209
 0.8831611  0.8865919  0.8987105  0.90113443 0.90174466 0.90874696
 0.9164089  0.92978346 0.9360894  0.94059396 0.94745183 0.95301783
 0.95748913 0.9608601  0.96132696 0.96468484 0.9626491  0.9606947
 0.9657568  0.9680273  0.97003    0.9699621  0.96952033 0.97039044
 0.97104406 0.97127604 0.97150177 0.971376   0.9730829  0.9749674
 0.9744505  0.97450405 0.9732757  0.9717718  0.9690493  0.96674824
 0.968257   0.96957976 0.96722746 0.9664414  0.9651397  0.9637256
 0.96426237 0.963984   0.961771